In [ ]:
import MeshFEM
import mesh_energy, panelization, py_newton_optimizer, benchmark, sim_utils
import mesh, parallelism, fd_validation, energy, viewer, loads
import numpy as np

In [ ]:
CLOSEST_PT_SPRINGS = False

In [ ]:
m = mesh.Mesh('../3rdparty/MeshFEM/misc/examples/meshes/lilium.msh', embeddingDimension=3)

In [ ]:
optVars = mesh_energy.NodalVars(m, 3) # Create per-node position variables (the variable dimension 3 here can also be inferred).
em = MeshFEM.EmbeddedMesh(m, optVars) # A wrapper object used to visualize the deformation described by `optVars`.

In [ ]:
# Create the objective terms
panelize = panelization.Panelization(m, optVars)
material = mesh_energy.MembraneMaterial(energy.NeoHookeanYoungPoisson(2, 1000, 0.3))
membrane = mesh_energy.NeoHookeanMembrane(m, optVars, material)
membrane.suppressSparsity = True # Negligible acceleration: membrane term Hessian sparsity is a subset of the hinge energy...

# Springs pulling vertices toward their original positions
if CLOSEST_PT_SPRINGS:
    import closest_point_projection
    proj = closest_point_projection.ClosestPointProjection(m)
    springs = loads.ProjectedSprings(optVars, np.arange(m.numVertices(), dtype=np.int32), proj, 1e3)
else:
    attachmentPoints = [loads.AttachmentPointCoordinate([i], [1]) for i in range(optVars.numVars())]
    targets = [loads.AttachmentPointCoordinate(v) for v in m.vertices().ravel()]
    springs = loads.Springs(optVars, attachmentPoints, targets, 1e3)

In [ ]:
# Configure a shape preservation term based on fitting elements' tangent planes to their initial planes.
tpf = panelization.TangentPlaneFitter(m, optVars, stiffness=2.5e3, variant=panelization.TangentPlaneFittingVariant.FitNormalOnly)

In [ ]:
# Combat shrinkage by fitting surface area to its initial value
saf = panelization.SurfaceAreaFitter(m, optVars)

In [ ]:
view = viewer.Viewer(em, wireframe=True)
view.setShadingType(viewer.ShadingType.FLAT)
view.setCameraParams(((3.9110356579312917, -3.8241043959992074, 2.167624091124389),
                      (-0.33101716247458735, 0.28436346407766794, 0.8997583333568081),
                      (0.047414196153991346, -0.1781858810189365, -0.40605702520247644)))
vtgt = viewer.Viewer(m, superView=view)
vtgt.makeTransparent(color='green')
view.show()

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(optVars, [panelize, membrane, tpf, saf])
prob.setCustomIterationCallback(view.updater(updateFrequency=5)) # Update every 5 iterations
opt = prob.optimizer() # Create a solver

In [ ]:
# Keep the mesh boundary vertices on the ground (but alllow them to slide in plane)
prob.setFixedVars(sim_utils.getBBoxVars(em, sim_utils.BBoxFace.MIN_Z, tol=1e-2, displacementComponents=[2]))

In [ ]:
prob.hessianShift = 1e-8

In [ ]:
def runWithSettings(delta, springStiffness = 1e3, verbose=False, maxIter = 100):
    panelize.materialForElement(0).delta = delta
    springs.setStiffnesses(springStiffness)
    opt.options.verbose = verbose
    opt.options.gradTol = 1e-6
    opt.options.niter = maxIter
    opt.optimize()

In [ ]:
prob.setWeights([1, 1, 1, 1e4]) # Increase surface area fitting weight.
saf.A_tgt = 2 # Fit to a surface area of `2` rather than the initial area

In [ ]:
saf.surfaceArea()

In [ ]:
benchmark.reset()
runWithSettings(0.1)
runWithSettings(0.05)
runWithSettings(0.025)
runWithSettings(0.005)
runWithSettings(0.0025)
runWithSettings(0.001)
benchmark.report()

In [ ]:
saf.surfaceArea()

In [ ]:
# Render springs; this currently cannot be used with the offscreen renderer/video writer
# if CLOSEST_PT_SPRINGS:
#     ns = springs.numSprings()
#     P = np.array([(springs.attachmentPointB(s).position, springs.attachmentPointB(s).preprojectedPosition) for s in range(ns)]).reshape(-1, 3)
#     E = np.column_stack((2 * np.arange(ns), 2 * np.arange(ns) + 1)) 
#     lv = viewer.Viewer((P, E), superView=view)
#     lv.showPoints()

In [ ]:
!mkdir -p results

In [ ]:
name = 'results/panelized_result' + ('_cp' if CLOSEST_PT_SPRINGS else '')

orender = view.offscreenRenderer(scale=4)
orender.orbitAnimation(name + '.mp4', 480, axis=[0, 0, 1], outWidth=1024, outHeight=1024, framerate=60, quality='-crf 16')

mesh.save(name + '.msh', em.embeddedVertices(), m.elements())